# Cognitive LLM — Phase 1: Colab T4 Debug Notebook

This notebook implements Phase 1 of the Cognitive LLM Architecture:
- Load SmolLM3 3B with 4-bit quantization
- Inject Block 1 (SurpriseGate) + Block 6 (HomeostaticNorm)
- Train 100 steps on GSM8K
- Verify loss is decreasing
- Evaluate on 50 GSM8K examples

In [ ]:
# Install dependencies
!pip install -q torch transformers datasets accelerate bitsandbytes peft trl wandb lm-eval pyyaml

In [ ]:
# Clone the repo (adjust URL as needed)
# !git clone https://github.com/YOUR_USERNAME/cognitive-llm.git
# %cd cognitive-llm

import sys
sys.path.insert(0, '.')

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

In [ ]:
# Load SmolLM3 3B with 4-bit quantization for T4 16GB
model_id = 'HuggingFaceTB/SmolLM3-3B'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map='auto',
    torch_dtype=torch.bfloat16,
)

print(f'Model loaded: {base_model.config.hidden_size}d, {base_model.config.num_hidden_layers} layers')

In [ ]:
# Wrap with Cognitive blocks (B1 + B6 only for Phase 1 debug)
from cognitive_llm.models.cognitive_model import CognitiveModel

config = {
    'use_block1': True,   # Surprise gate ON
    'use_block2': False,  # Memory OFF for now
    'use_block3': False,
    'use_block4': False,
    'use_block5': False,
    'use_block6': True,   # HomeostaticNorm always ON
}

model = CognitiveModel(base_model, config)
print('CognitiveModel created successfully')

# Quick forward pass test
test_ids = tokenizer('Hello world', return_tensors='pt').input_ids.to('cuda')
with torch.no_grad():
    outputs = model(test_ids)
print(f'Logits shape: {outputs["logits"].shape}')
print('Forward pass OK!')

In [ ]:
# Load GSM8K dataset
from datasets import load_dataset

dataset = load_dataset('gsm8k', 'main', split='train[:500]')
print(f'GSM8K samples: {len(dataset)}')
print(f'Example: {dataset[0]["question"][:100]}...')

In [ ]:
# Tokenize dataset
def tokenize_gsm8k(examples):
    texts = [
        f'Question: {q}\nAnswer: {a}'
        for q, a in zip(examples['question'], examples['answer'])
    ]
    tokenized = tokenizer(
        texts,
        truncation=True,
        max_length=512,
        padding='max_length',
        return_tensors='pt',
    )
    tokenized['labels'] = tokenized['input_ids'].clone()
    return tokenized

tokenized_dataset = dataset.map(
    tokenize_gsm8k,
    batched=True,
    remove_columns=dataset.column_names,
)
tokenized_dataset.set_format('torch')

from torch.utils.data import DataLoader
train_loader = DataLoader(tokenized_dataset, batch_size=2, shuffle=True)

In [ ]:
# Train 100 steps
from cognitive_llm.training.trainer import CognitiveTrainer

training_config = {
    'learning_rate': 2e-4,
    'max_steps': 100,
    'warmup_steps': 10,
    'gradient_accumulation': 8,
    'max_grad_norm': 1.0,
    'eval_every_n_steps': 50,
    'save_every_n_steps': 100,
    'use_wandb': False,  # Set to True if you have wandb configured
}

trainer = CognitiveTrainer(
    model=model,
    train_dataloader=train_loader,
    config=training_config,
)

losses = trainer.train()
print(f'Training complete! Final loss: {losses[-1]:.4f}')

In [ ]:
# Plot loss curve — must be decreasing
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
plt.plot(losses)
plt.xlabel('Step')
plt.ylabel('Loss')
plt.title('Cognitive LLM Training Loss (B1+B6, GSM8K)')
plt.grid(True)
plt.show()

# Verify loss is decreasing
if len(losses) > 10:
    first_10 = sum(losses[:10]) / 10
    last_10 = sum(losses[-10:]) / 10
    print(f'First 10 avg: {first_10:.4f}, Last 10 avg: {last_10:.4f}')
    if last_10 < first_10:
        print('Loss is DECREASING — training is working!')
    else:
        print('WARNING: Loss is NOT decreasing — check configuration!')

In [ ]:
# Quick baseline scoring on 50 GSM8K examples
test_dataset = load_dataset('gsm8k', 'main', split='test[:50]')

correct = 0
total = 0

model.train(False)
for example in test_dataset:
    prompt = f"Question: {example['question']}\nAnswer:"
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=512).to('cuda')
    
    with torch.no_grad():
        generated = base_model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
        )
    
    response = tokenizer.decode(generated[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    
    # Extract answer
    import re
    pred_match = re.search(r'####\s*([-\d,\.]+)', response)
    target_match = re.search(r'####\s*([-\d,\.]+)', example['answer'])
    
    if pred_match and target_match:
        if pred_match.group(1).replace(',', '') == target_match.group(1).replace(',', ''):
            correct += 1
    total += 1

print(f'GSM8K accuracy (50 samples): {correct}/{total} = {correct/total*100:.1f}%')